# Player Tracking in Sports - Multi-View Tracking and 3D Reconstruction

## Imports

In [ ]:
from src.utils.video import open_video, get_frames, save_video
from src.tracking.motion_detection import MOG2_motion_detection, refine_blobs
from src.tracking.image_processing import opening_closing, remove_reflections
from src.utils.visualization import show_image
from src.tracking.yolo_tracking import run_yolo_tracking, extract_tracking_data

from time import time
import cv2
import logging
from ultralytics import YOLO


# Suppress YOLO logging
logging.getLogger("ultralytics").setLevel(logging.WARNING)
logging.getLogger("ultralytics.hub.utils").setLevel(logging.WARNING)

## Path Definitions

In [ ]:
VIDEOS_DIR = "data/videos"

# Define fundamental paths for each camera
CAMERAS = {
    "cam_13": {
        "video_path": f"{VIDEOS_DIR}/out13.mp4",
    },
    "cam_2": {
        "video_path": f"{VIDEOS_DIR}/out2.mp4",
    },
    "cam_4": {
        "video_path": f"{VIDEOS_DIR}/out4.mp4",
    },
}

## Open Video and Read Frames

In [ ]:
# Currently just one camera
cap = open_video(CAMERAS["cam_13"]["video_path"])
frames_color, _ = get_frames(cap, max_frames=None)

# Release the video capture object to free resources
cap.release()

show_image(frames_color[0], title="Original Frame")

## Image Preprocessing

We first apply a Gaussian blur to reduce reflection and noise in the frames. This helps improve the performance of motion detection algorithms.

Then, we apply a custom function `remove_reflections` to further clean the frames by removing reflections and shadows that can interfere with motion detection.

In [ ]:
# Function to remove reflections and shadows from the motion masks
refined_frames_color = remove_reflections([frames_color[0]], sat_threshold=80, val_threshold=180, inpaint_radius=5)

show_image(refined_frames_color[0], title="Preprocessed Frame")

## Motion Detection

### Mixture of Gaussians (MOG2) Background Subtraction

In [ ]:
# Parameters for MOG2 motion detection
learning_rate = -1 # Use default learning rate (auto-adaptive)
history_length = 1500
var_threshold = 16
detect_shadows = False

# Compute motion masks using MOG2
masks = MOG2_motion_detection(frames_color, learning_rate, history_length, var_threshold, detect_shadows)

### Motion Video Post-processing

- Apply morphological operations to clean up the masks.
- Blob filtering to remove small noise blobs and keep only significant motion areas.

In [ ]:
# Morphological opening and closing to clean up the masks
opening_kernel_size = 7
closing_kernel_size = 7
masks = opening_closing(masks, opening_kernel_size=opening_kernel_size, closing_kernel_size=closing_kernel_size)

# Blob filtering to remove small noise blobs and keep only significant motion areas
masks = refine_blobs(masks, min_area=200)

In [ ]:
 # Save results as video
save_video(masks, "results/motion_detection/cam13/cam_13_mog2_masks.mp4")

**Temporary** : Cam 13 — Region of Interest

In [ ]:
# Adjust roi_y to find the right horizontal boundary for cam 13
roi_y = 300
frame = frames_color[0]  # Use the first frame for visualization
h, w, _ = frame.shape

cv2.line(frame, (0, roi_y), (w, roi_y), (255, 255, 255), 2)

## Using YOLOv8 for Player Detection
We could also use a deep learning-based object detection model like YOLOv8 to detect players in each frame. This can be done by loading a pre-trained YOLOv8 model and running inference on the frames to get bounding boxes for detected players.

In [ ]:
model = YOLO("yolov8n.pt") # Load the pre-trained YOLOv8n model

### Run YOLO Tracking on All Frames

Use YOLOv8's built-in `.track()` method to detect objects and maintain track IDs across frames. The `persist=True` parameter ensures consistent tracking IDs throughout the video sequence.

**Output per frame:**
- Bounding boxes: `[x1, y1, x2, y2]` (pixel coordinates)
- Track IDs: Unique identifier per object across frames
- Confidence scores: Detection confidence (0–1)
- Class labels: What was detected (e.g., "person", "sports_ball")

In [ ]:
print(f"Processing {len(frames_color)} frames with YOLO tracking...")
start_time = time.time()

tracked_results = run_yolo_tracking(model, frames_color)

end_time = time.time()
print(f"Tracking complete")
print(f"Total detections: {sum(len(r.boxes) for r in tracked_results)}")
print(f"Processing time: {end_time - start_time:.2f} seconds")

## Extract Raw Detection Data

Parse all tracking results into a structured format containing bounding boxes, track IDs, confidence scores, and class labels for each frame.

In [ ]:
tracking_data = extract_tracking_data(tracked_results, model)

# Show sample from frame 0
frame_0_data = tracking_data[0]

print("Sample detections from frame 0:")
print(f"  Total detections: {frame_0_data['num_detections']}")

for det in frame_0_data["detections"][:3]:  # Show first 3
    print(f"  Track ID: {det['track_id']}, Class: {det['class_name']}, Conf: {det['confidence']}, BBox: {det['bbox']}")
else:
    print("\nNo detections in frame 0")

## Visualize Tracking Results

Draw bounding boxes and track IDs on sample frames to verify tracking quality. YOLO's `.plot()` method automatically adds colored boxes with class labels.

In [ ]:
# Visualize tracking results on sample frames
sample_frames = [0, len(frames_color)//2, len(frames_color)-1]  # First, middle, last frame

for frame_idx in sample_frames:
    result = tracked_results[frame_idx]
    
    # Get the annotated frame with bounding boxes (YOLO's built-in visualization)
    annotated_frame = result.plot()
    
    # Display the frame
    title = f"Frame {frame_idx}: {len(result.boxes)} detections"
    show_image(annotated_frame, title=title)
    
    # Print detection info for this frame
    if len(result.boxes) > 0:
        print(f"\n{title}")
        for i in range(min(5, len(result.boxes))):  # Show first 5 detections
            track_id = int(result.boxes.id[i].item()) if result.boxes.id is not None else None
            conf = float(result.boxes.conf[i].item())
            class_id = int(result.boxes.cls[i].item())
            class_name = model.names[class_id]
            print(f"  [{i}] Track ID: {track_id}, Class: {class_name}, Confidence: {conf:.3f}")
    else:
        print(f"\n{title} - No detections")